In [1]:
import os
import re
from collections import defaultdict, Counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import polars as pl
import loguru as logger

from settings.settings_analyze_efizz import Settings_ae
from behave_analysis.process.session import get_experiment
from behave_analysis.utils.rayleigh.load_rayleigh import collect_all_rayleigh_paths, load_all_rayleigh_data
from behave_analysis.utils.rayleigh.manipulate_rayleigh_df import extract_compartment_values, extract_firing_rates
from behave_analysis.utils.creating_directories import make_directory
from settings.settings_analyze_efizz import Settings_ae as Settings
from behave_analysis.analyze.TunED.model import TunEdModel

In [2]:
from behave_analysis.database.Experiments.JAL003_ex import JAL3_25aug, JAL3_1sept, JAL3_4sept, JAL3_7sept
from behave_analysis.database.Experiments.JAL004_ex import JAL4_3rdSept, JAL4_19thSept, JAL4_28aug, JAL4_11thSept
from behave_analysis.database.Experiments.JAL005_ex import JAL005_8thSept, JAL005_21stSept
from behave_analysis.database.Experiments.JAL006_ex import JAL6_28mar, JAL6_flip4_21mar, JAL6_flip5_25mar, JAL6_flip3_18mar, JAL6_flip7_1apr
from behave_analysis.database.Experiments.JAL007_ex import JAL7_sesh8_9apr, JAL7_sesh9_16apr, JAL7_flip5_22mar, JAL7_flip2_12mar, JAL7_23apr, JAL7_30apr
from behave_analysis.database.Experiments.JAL008_ex import JAL8_flip1_25apr, JAL8_flip2_29apr, JAL8_tiny_3may, JAL8_flip4_10may, JAL8_14may, JAL8_21may

In [3]:
experiments_objects = [JAL6_flip7_1apr, JAL6_flip3_18mar, JAL6_flip4_21mar, JAL6_flip5_25mar, JAL6_28mar,
                       JAL3_25aug, JAL3_1sept, JAL3_4sept, JAL3_7sept,
                       JAL005_8thSept, JAL005_21stSept,
                       JAL7_sesh8_9apr, JAL7_sesh9_16apr, JAL7_flip5_22mar, JAL7_flip2_12mar, JAL7_23apr,
                       JAL8_flip1_25apr, JAL8_flip2_29apr, JAL8_flip4_10may, JAL8_14may,
                       JAL4_3rdSept, JAL4_19thSept, JAL4_28aug, JAL4_11thSept]

session_names = ["JAL6_flip7_1apr", "JAL6_flip3_18mar", "JAL6_flip4_21mar", "JAL6_flip5_25mar", "JAL6_28mar",
                 "JAL3_25aug", "JAL3_1sept", "JAL3_4sept", "JAL3_7sept", "JAL005_8thSept", "JAL005_21stSept",
                 "JAL5_8thSept", "JAL5_21stSept",
                 "JAL7_sesh8_9apr", "JAL7_sesh9_16apr", "JAL7_flip5_22mar", "JAL7_flip2_12mar", "JAL7_23apr",
                 "JAL8_flip1_25apr", "JAL8_flip2_29apr", "JAL8_flip4_10may", "JAL8_14may",
                 "JAL4_3rdSept", "JAL4_19thSept", "JAL4_28aug", "JAL4_11thSept"]

tinny_barrier = [JAL8_tiny_3may, JAL8_21may, JAL7_30apr]

# Mice groups based on session names
mice_groups = {
    "JAL6": ['JAL6_flip7_1apr', 'JAL6_flip3_18mar', 'JAL6_flip4_21mar', 'JAL6_flip5_25mar', 'JAL6_28mar'],
    "JAL3": ['JAL3_25aug', 'JAL3_1sept', 'JAL3_4sept', 'JAL3_7sept'],
    "JAL7": ['JAL7_sesh8_9apr', 'JAL7_sesh9_16apr', 'JAL7_flip5_22mar', 'JAL7_flip2_12mar', 'JAL7_23apr'],
    "JAL8": ['JAL8_flip1_25apr', 'JAL8_flip2_29apr', 'JAL8_flip4_10may', 'JAL8_14may'],
    "JAL4": ['JAL4_3rdSept', 'JAL4_19thSept', 'JAL4_28aug', 'JAL4_11thSept'],
    "JAL5": ['JAL5_8thSept', 'JAL5_21stSept']}



In [4]:
def regex(angle):
    "Remove unwanted characters from angle file string"
    pattern = r'^(.*?)(?=_Rayleigh)'
    match = re.search(pattern, angle)
    assert match, f"Could not find match for {angle}"
    return match.group(0)

def nest_dic():
    """A function to create arbitrarily nested dictionaries"""
    return defaultdict(nest_dic)

In [5]:
# Init Params
conditions = ["shelter_only", "barrier_pre_flip", "barrier_post_flip"]
total_cells = 0
total_sessions = 0
rayleigh_threshold = 0.15
fr_threshold = 5 # Hz
dir = make_directory(r"Z:\Jasmine_Laurence\rayleigh_analysis")

In [6]:
angle_keys = ['hdir_Rayleigh.arrow', 'hsa_Rayleigh.arrow', 'h_postflipbar_a_Rayleigh.arrow', 'h_preflipbar_a_Rayleigh.arrow']

# Just threat zone

In [7]:
# TODO firing rate threshold not implemented

threat_dict = nest_dic() # dict[session][cell][condition] = {max_rayleigh_angle}
cell_count = 0
not_meet_threshold = 0

for i, session in enumerate(experiments_objects):
    loaded_session = get_experiment(session)
    paths = collect_all_rayleigh_paths(session = loaded_session, cluster_type = "good", conditions= conditions) # paths[condition][angles]
    condition_data = load_all_rayleigh_data(paths) # condition_data[condition][angles] ~ hdir, hsa, center, post flip, pre flip
    
    # Count the number of sessions and cells
    nCells = len(condition_data["shelter_only"]["hdir_Rayleigh.arrow"]["Rayleigh"]) # Cells are the same for all conditions and angles in one session
    cell_count += nCells
    
    for cell in range(nCells):
        tuned = False # A pointer to check if the cell is tuned to any angle in any condition

        for ci, condition in enumerate(condition_data.keys()):
            rayleigh = 0 # Per condition reset the rayleigh value
            for angle in angle_keys:
                
                #Just the threat zone
                output = extract_compartment_values(condition_data[condition][angle], column_name="Rayleigh") # Extract the rayleigh values for THREAT ONLY
                if np.logical_and(output[cell][1] > rayleigh, output[cell][1] > rayleigh_threshold):
                    rayleigh = output[cell][1]
                    max_angle_str = regex(angle)
                    tuned = True
                    
            # For each cell in each session, save the max rayleigh values and corresponding condition | angle combo
            try:
                if tuned:
                    threat_dict[i][cell][condition] = max_angle_str
                else:
                    threat_dict[i][cell][condition] = "Not tuned"
            except:
                print(f"Cell {cell} in session {i} in {condition} did not meet threshold")
                
        if not tuned:
            # Track the number of cells that did not meet the threshold
            not_meet_threshold += 1
    

In [8]:
print(not_meet_threshold)
print(cell_count)

1625
4858


# Count between conditions

In [149]:
TC_shelter_to_barrier = defaultdict(int)
TC_barrier_to_flipped = defaultdict(int)
labels = {'hdir', 'h_preflipbar_a', 'Not tuned', 'hsa', 'h_postflipbar_a'}
shelter_2_barrier_matrix = pd.DataFrame(0, index=sorted(labels), columns=sorted(labels)) # intialize the matrix
barrier_2_flipped_matrix = pd.DataFrame(0, index=sorted(labels), columns=sorted(labels)) # intialize the matrix

for session in threat_dict:
    for cell in threat_dict[session]:
        x = threat_dict[session][cell]["shelter_only"] # "hsa"
        y = threat_dict[session][cell]["barrier_pre_flip"] # "hdir"
        z = threat_dict[session][cell]["barrier_post_flip"] # "h_postflipbar_a"
        shelter_2_barrier_matrix.loc[x, y] += 1
        barrier_2_flipped_matrix.loc[y, z] += 1

print(shelter_2_barrier_matrix)
print(barrier_2_flipped_matrix)

                 Not tuned  h_postflipbar_a  h_preflipbar_a  hdir  hsa
Not tuned             1752               50             111   111   83
h_postflipbar_a          0              349             194   127   94
h_preflipbar_a           0              132             448   128   94
hdir                     0               50              87   417   69
hsa                      0               92             130   111  229
                 Not tuned  h_postflipbar_a  h_preflipbar_a  hdir  hsa
Not tuned             1625               51              14    29   33
h_postflipbar_a          0              469              75    74   55
h_preflipbar_a           0              147             531   164  128
hdir                     0              155              52   601   86
hsa                      0              107              56   111  295


# Plot the matrices in counts

In [73]:
# Plot the two matrices side by side
plt.figure(figsize=(16, 8))

# Plot the first heatmap (shelter to barrier)
plt.subplot(1, 2, 1)
mask1 = np.zeros_like(shelter_2_barrier_matrix, dtype=bool)
mask1[shelter_2_barrier_matrix.index.get_loc('Not tuned'), shelter_2_barrier_matrix.columns.get_loc('Not tuned')] = True
ax1 = sns.heatmap(shelter_2_barrier_matrix, annot=True, fmt='.01f', cmap='viridis', mask=mask1, cbar=True, square=True, linewidths=.5)
ax1.text(shelter_2_barrier_matrix.columns.get_loc('Not tuned') + 0.5, shelter_2_barrier_matrix.index.get_loc('Not tuned') + 0.5,
         f'{shelter_2_barrier_matrix.at["Not tuned", "Not tuned"]}', color='black', weight='bold', ha='center', va='center')
plt.title('Shelter to barrier transition')
plt.xlabel('Barrier Condition', fontsize=18)
plt.ylabel('Shelter Condtion', fontsize=18)

# Plot the second heatmap (barrier to flipped)
plt.subplot(1, 2, 2)
mask2 = np.zeros_like(barrier_2_flipped_matrix, dtype=bool)
mask2[barrier_2_flipped_matrix.index.get_loc('Not tuned'), barrier_2_flipped_matrix.columns.get_loc('Not tuned')] = True
ax2 = sns.heatmap(barrier_2_flipped_matrix, annot=True, fmt='.01f', cmap='viridis', mask=mask2, cbar=True, square=True, linewidths=.5)
ax2.text(barrier_2_flipped_matrix.columns.get_loc('Not tuned') + 0.5, barrier_2_flipped_matrix.index.get_loc('Not tuned') + 0.5,
         f'{barrier_2_flipped_matrix.at["Not tuned", "Not tuned"]}', color='black', weight='bold', ha='center', va='center')
plt.title('Barrier to flipped transition')
plt.xlabel('Flipped Barrier Condition', fontsize=18)
plt.ylabel('Barrier Condtion', fontsize=18)

plt.suptitle(f'Transition of Tuned Cells between Conditions for the Threat Zone Compartment \n Cell Count {cell_count} ', fontsize=20)
plt.tight_layout()
plt.show()


# Compute conditional probabilities

In [150]:
# P(A|B) = P(A and B) / P(B)

# P(A and B)
shelter_2_barrier_joint = shelter_2_barrier_matrix / cell_count
barrier_2_flipped_joint = barrier_2_flipped_matrix / cell_count


# P(B) marginal probability
shelter_2_barrier_marginal = shelter_2_barrier_joint.sum(axis=1)
barrier_2_flipped_marginal = barrier_2_flipped_joint.sum(axis=1)


# Conditional probabilities
shelter_2_barrier_conditional = shelter_2_barrier_joint.div(shelter_2_barrier_marginal, axis=0)
barrier_2_flipped_conditional = barrier_2_flipped_joint.div(barrier_2_flipped_marginal, axis=0)

# Assert that the sum of the rows is 1
assert np.allclose(shelter_2_barrier_conditional.sum(axis=1), 1)


In [151]:
# Plot the two matrices side by side
plt.figure(figsize=(16, 8))

# Plot the first heatmap (shelter to barrier)
plt.subplot(1, 2, 1)
mask1 = np.zeros_like(shelter_2_barrier_conditional, dtype=bool)
mask1[shelter_2_barrier_conditional.index.get_loc('Not tuned'), shelter_2_barrier_conditional.columns.get_loc('Not tuned')] = True
ax1 = sns.heatmap(shelter_2_barrier_conditional, annot=True, fmt='.02f', cmap='viridis', mask=mask1, cbar=True, square=True, linewidths=.5)
ax1.text(shelter_2_barrier_conditional.columns.get_loc('Not tuned') + 0.5, shelter_2_barrier_conditional.index.get_loc('Not tuned') + 0.5,
         f'{shelter_2_barrier_conditional.at["Not tuned", "Not tuned"]:.1f}', color='black', weight='bold', ha='center', va='center')
plt.title('Shelter to barrier transition \n P(Barrier | Shelter) ')
plt.xlabel('Barrier Condition', fontsize=18)
plt.ylabel('Shelter Condtion', fontsize=18)

# Plot the second heatmap (barrier to flipped)
plt.subplot(1, 2, 2)
mask2 = np.zeros_like(barrier_2_flipped_conditional, dtype=bool)
mask2[barrier_2_flipped_conditional.index.get_loc('Not tuned'), barrier_2_flipped_conditional.columns.get_loc('Not tuned')] = True
ax2 = sns.heatmap(barrier_2_flipped_conditional, annot=True, fmt='.02f', cmap='viridis', mask=mask2, cbar=True, square=True, linewidths=.5)
ax2.text(barrier_2_flipped_conditional.columns.get_loc('Not tuned') + 0.5, barrier_2_flipped_conditional.index.get_loc('Not tuned') + 0.5,
         f'{barrier_2_flipped_conditional.at["Not tuned", "Not tuned"]:.1f}', color='black', weight='bold', ha='center', va='center')
plt.title('Barrier to flipped transition\n P(Flipped Barrier | Barrier)')
plt.xlabel('Flipped Barrier Condition', fontsize=18)
plt.ylabel('Barrier Condtion', fontsize=18)

plt.suptitle(f'Transition P(Y|X) of Tuned Cells between Conditions for the Threat Zone Compartment \n Where X is the preceeding condition', fontsize=20)
plt.tight_layout()
plt.show()

# ------------------

# Just shelter compartment

In [89]:
# TODO firing rate threshold not implemented

shelter_dict = nest_dic() # dict[session][cell][condition] = {max_rayleigh_angle}
cell_count = 0
not_meet_threshold = 0

for i, session in enumerate(experiments_objects):
    loaded_session = get_experiment(session)
    paths = collect_all_rayleigh_paths(session = loaded_session, cluster_type = "good", conditions= conditions) # paths[condition][angles]
    condition_data = load_all_rayleigh_data(paths) # condition_data[condition][angles] ~ hdir, hsa, center, post flip, pre flip
    
    # Count the number of sessions and cells
    nCells = len(condition_data["shelter_only"]["hdir_Rayleigh.arrow"]["Rayleigh"]) # Cells are the same for all conditions and angles in one session
    cell_count += nCells
    
    for cell in range(nCells):
        tuned = False # A pointer to check if the cell is tuned to any angle in any condition

        for ci, condition in enumerate(condition_data.keys()):
            rayleigh = 0 # Per condition reset the rayleigh value
            for angle in angle_keys:
                
                #Just the threat zone
                # output = extract_compartment_values(condition_data[condition][angle], column_name="Rayleigh") # Extract the rayleigh values for THREAT ONLY
                # if np.logical_and(output[cell][1] > rayleigh, output[cell][1] > rayleigh_threshold):
                #     rayleigh = output[cell][1]
                #     max_angle_str = regex(angle)
                #     tuned = True
                    
                # Just the shelter
                output = extract_compartment_values(condition_data[condition][angle], column_name="Rayleigh") # Extract the rayleigh values for SHELTER ONLY
                if np.logical_and(output[cell][0] > rayleigh, output[cell][0] > rayleigh_threshold):
                    rayleigh = output[cell][0]
                    max_angle_str = regex(angle)
                    tuned = True

                # # Whole arena rayleigh
                # output = condition_data[condition][angle]["arena_rayleigh"] # Whole arena rayleigh
                # if np.logical_and(output[cell] > rayleigh, output[cell] > rayleigh_threshold):
                #     rayleigh = output[cell]
                #     max_angle_str = regex(angle)
                #     tuned = True
                    
            # For each cell in each session, save the max rayleigh values and corresponding condition | angle combo
            try:
                if tuned:
                    shelter_dict[i][cell][condition] = max_angle_str
                else:
                    shelter_dict[i][cell][condition] = "Not tuned"
            except:
                print(f"Cell {cell} in session {i} in {condition} did not meet threshold")
                
        if not tuned:
            # Track the number of cells that did not meet the threshold
            not_meet_threshold += 1
    

Exception ignored in: <function Image.__del__ at 0x000002B422FD0F70>
Traceback (most recent call last):
  File "c:\Users\laurence\miniconda3\envs\subgoal\lib\tkinter\__init__.py", line 4017, in __del__
    self.tk.call('image', 'delete', self.name)
RuntimeError: main thread is not in main loop
Exception ignored in: <function Variable.__del__ at 0x000002B422FB4670>
Traceback (most recent call last):
  File "c:\Users\laurence\miniconda3\envs\subgoal\lib\tkinter\__init__.py", line 363, in __del__
    if self._tk.getboolean(self._tk.call("info", "exists", self._name)):
RuntimeError: main thread is not in main loop
Exception ignored in: <function Variable.__del__ at 0x000002B422FB4670>
Traceback (most recent call last):
  File "c:\Users\laurence\miniconda3\envs\subgoal\lib\tkinter\__init__.py", line 363, in __del__
    if self._tk.getboolean(self._tk.call("info", "exists", self._name)):
RuntimeError: main thread is not in main loop
Exception ignored in: <function Variable.__del__ at 0x00000

# across session for shelter condition

In [90]:
TC_shelter_to_barrier = defaultdict(int)
TC_barrier_to_flipped = defaultdict(int)
labels = {'hdir', 'h_preflipbar_a', 'Not tuned', 'hsa', 'h_postflipbar_a'}
shelter_2_barrier_matrix = pd.DataFrame(0, index=sorted(labels), columns=sorted(labels)) # intialize the matrix
barrier_2_flipped_matrix = pd.DataFrame(0, index=sorted(labels), columns=sorted(labels)) # intialize the matrix

for session in threat_dict:
    for cell in threat_dict[session]:
        x = shelter_dict[session][cell]["shelter_only"] # "hsa"
        y = shelter_dict[session][cell]["barrier_pre_flip"] # "hdir"
        z = shelter_dict[session][cell]["barrier_post_flip"] # "h_postflipbar_a"
        shelter_2_barrier_matrix.loc[x, y] += 1
        barrier_2_flipped_matrix.loc[y, z] += 1

print(shelter_2_barrier_matrix)
print(barrier_2_flipped_matrix)

                 Not tuned  h_postflipbar_a  h_preflipbar_a  hdir  hsa
Not tuned             2130               39              80    47   38
h_postflipbar_a          0              306             162    76   64
h_preflipbar_a           0              112             398    71   84
hdir                     0               45              67   349   69
hsa                      0               53             128    66  474
                 Not tuned  h_postflipbar_a  h_preflipbar_a  hdir  hsa
Not tuned             2008               30              45    24   23
h_postflipbar_a          0              354              82    54   65
h_preflipbar_a           0              151             496    89   99
hdir                     0               84              51   428   46
hsa                      0               91              72    50  516


In [146]:
# P(A|B) = P(A and B) / P(B)

# P(A and B)
shelter_2_barrier_joint = shelter_2_barrier_matrix / cell_count
barrier_2_flipped_joint = barrier_2_flipped_matrix / cell_count


# P(B) marginal probability
shelter_2_barrier_marginal = shelter_2_barrier_joint.sum(axis=1)
barrier_2_flipped_marginal = barrier_2_flipped_joint.sum(axis=1)


# Conditional probabilities
shelter_2_barrier_conditional = shelter_2_barrier_joint.div(shelter_2_barrier_marginal, axis=0)
barrier_2_flipped_conditional = barrier_2_flipped_joint.div(barrier_2_flipped_marginal, axis=0)

# Assert that the sum of the rows is 1
assert np.allclose(shelter_2_barrier_conditional.sum(axis=1), 1)

In [144]:
# Plot the two matrices side by side
plt.figure(figsize=(16, 8))

# Plot the first heatmap (shelter to barrier)
plt.subplot(1, 2, 1)
mask1 = np.zeros_like(shelter_2_barrier_conditional, dtype=bool)
mask1[shelter_2_barrier_conditional.index.get_loc('Not tuned'), shelter_2_barrier_conditional.columns.get_loc('Not tuned')] = True
ax1 = sns.heatmap(shelter_2_barrier_conditional, annot=True, fmt='.02f', cmap='viridis', mask=mask1, cbar=True, square=True, linewidths=.5)
ax1.text(shelter_2_barrier_conditional.columns.get_loc('Not tuned') + 0.5, shelter_2_barrier_conditional.index.get_loc('Not tuned') + 0.5,
         f'{shelter_2_barrier_conditional.at["Not tuned", "Not tuned"]:.1f}', color='black', weight='bold', ha='center', va='center')
plt.title('Shelter to barrier transition \n P(Barrier | Shelter) ')
plt.xlabel('Barrier Condition', fontsize=18)
plt.ylabel('Shelter Condtion', fontsize=18)

# Plot the second heatmap (barrier to flipped)
plt.subplot(1, 2, 2)
mask2 = np.zeros_like(barrier_2_flipped_conditional, dtype=bool)
mask2[barrier_2_flipped_conditional.index.get_loc('Not tuned'), barrier_2_flipped_conditional.columns.get_loc('Not tuned')] = True
ax2 = sns.heatmap(barrier_2_flipped_conditional, annot=True, fmt='.02f', cmap='viridis', mask=mask2, cbar=True, square=True, linewidths=.5)
ax2.text(barrier_2_flipped_conditional.columns.get_loc('Not tuned') + 0.5, barrier_2_flipped_conditional.index.get_loc('Not tuned') + 0.5,
         f'{barrier_2_flipped_conditional.at["Not tuned", "Not tuned"]:.1f}', color='black', weight='bold', ha='center', va='center')
plt.title('Barrier to flipped transition\n P(Flipped Barrier | Barrier)')
plt.xlabel('Flipped Barrier Condition', fontsize=18)
plt.ylabel('Barrier Condtion', fontsize=18)

plt.suptitle(f'Transition P(Y|X) of Tuned Cells between Conditions for the Shelter Zone Compartment \n Where X is the preceeding condition', fontsize=20)
plt.tight_layout()
plt.show()